# Cartographie prédictive GWS — HuberRegressor

Notebook Google Colab généré à partir du script Python d'origine : **`HuberRegressor.py`**

**Dossier de données attendu sur Google Drive :** `/content/drive/MyDrive/input_data_ML`
(il doit contenir `name_files.txt` et les rasters listés dedans).

**Ordre d'exécution :** lancer les cellules de haut en bas (*Exécution → Tout exécuter*).
Chaque cellule exécute directement son étape, comme le script d'origine (qui n'a pas de fonction `main()`).
Les résultats sont écrits dans `input_data_ML/GWS_PREDICTIONS_GEE_HUBER` sur Drive.

Le code du script est repris tel quel. Les seules lignes ajoutées ou modifiées pour Colab
sont repérées par le commentaire `# [COLAB]`.

### Description (en-tête du script d'origine)

```text
# ============================================================
#  CARTOGRAPHIE PREDICTIVE GWS
#  (TRAIN = longue période, TEST = 5% des dates, CLIP GEE, SANS SOUS-ÉCHANTILLONNAGE)
#
#  - Modèle HuberRegressor (régression robuste) sur grille GLDAS (~25 km)
#
#  - Variables explicatives :
#       * Precipitations_IDW (ou Precipitations_API_IDW)  → précipitations (journalier)
#       * MODIS_LST_Clipped                              → LST MODIS (journalier)
#       * MODIS_NDVI_Clipped                             → NDVI MODIS (16 jours, étendu)
#       * MODIS_ET_Clipped                               → ET MODIS (~8 jours, étendu)
#       * ERA5_Qsb_Clipped                               → Qsb ERA5 (journalier)
#       * SMAP_RZSM_Clipped                              → RZSM SMAP (journalier)
#       * SMAP_SoilM_Clipped                             → Soil Moisture SMAP (journalier)
#
#  - Cible :
#       * GLDAS_GWS_Clipped                              → GWS (GLDAS)
#
#  - Features finales :
#       [P, LST, NDVI_ext, ET_ext, Qsb, RZSM, SoilM, sinDOY, cosDOY]
#
#  - Rééchantillonnage en respectant correctement les NoData
#  - Remplissage des NaN d'entrée → cartes SANS TROUS à l’intérieur des Regions
#  - Clip de toutes les cartes (test + forecast) sur la zone Regions
#
#  ➤ Périodes utilisées :
#    TRAIN pool : 2021-01-01 → 2025-06-30
#    TEST = 5% des dates tirées aléatoirement dans cette fenêtre
#    FORECAST : 2025-07-01 → 2025-10-30
# ============================================================
```

## A. Préparation de l'environnement Colab

### A.1 Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### A.2 Installer les dépendances

`rasterio` n'est pas préinstallé sur Colab (scikit-learn, numpy et joblib le sont).

In [ ]:
!pip install -q rasterio

### A.3 Imports

In [ ]:
import os
import re
import time
import bisect
from datetime import datetime, date

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import HuberRegressor

# [COLAB] Versions des bibliothèques utilisées
from importlib.metadata import version as _pkg_version
for _pkg in ('numpy', 'scikit-learn', 'rasterio'):
    print(f"{_pkg:13s}: {_pkg_version(_pkg)}")

## 1. Paramètres généraux

`BASE_DIR` pointe vers le dossier `input_data_ML` de votre Drive.

In [ ]:
# ------------------------------------------------------------
# 1) Paramètres généraux
# ------------------------------------------------------------
np.random.seed(42)

# [COLAB] Dossier d'entrée sur Google Drive (remplace C:\Users\PC\Downloads\input_data_ML)
BASE_DIR = "/content/drive/MyDrive/input_data_ML"
NAME_FILE = os.path.join(BASE_DIR, "name_files.txt")

# [COLAB] Arrêt immédiat si Drive n'est pas monté ou si le dossier est mal placé
# (sinon le dossier de sortie serait créé hors de Drive et perdu en fin de session)
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(
        f"Dossier introuvable: {BASE_DIR} -> monter Drive (cellule A.1) et vérifier "
        "que input_data_ML est bien à la racine de 'Mon Drive'."
    )
if not os.path.isfile(NAME_FILE):
    raise FileNotFoundError(f"Fichier introuvable: {NAME_FILE}")

OUT_DIR = os.path.join(BASE_DIR, "GWS_PREDICTIONS_GEE_HUBER")
os.makedirs(OUT_DIR, exist_ok=True)

NODATA_VALUE = -9999.0
TEST_FRACTION = 0.05

# 👉 Choix de la source de précipitations
# False → Precipitations_IDW (si existe)
# True  → Precipitations_API_IDW (si existe)
PREFER_API_PRECIP = False

## 2. Fenêtres temporelles TRAIN / FORECAST

In [ ]:
# ------------------------------------------------------------
# 2) Fenêtres temporelles TRAIN / FORECAST
# ------------------------------------------------------------
TRAIN_START = date(2021, 1, 1)
TRAIN_END   = date(2025, 6, 30)

FORECAST_START = datetime(2025, 7, 1)
FORECAST_END   = datetime(2025, 10, 30)

print("=== PARAMÈTRES GÉNÉRAUX ===")
print(" BASE_DIR :", BASE_DIR)
print(" NAME_FILE :", NAME_FILE)
print(" OUT_DIR :", OUT_DIR)
print(" TRAIN_START/END :", TRAIN_START, "→", TRAIN_END)
print(" TEST_FRACTION :", TEST_FRACTION)
print(" FORECAST_START/END :", FORECAST_START.date(), "→", FORECAST_END.date())
print(" PREFER_API_PRECIP :", PREFER_API_PRECIP)
print("============================\n")

### 2.1 Dates exclues (LST manquante)

In [ ]:
# ------------------------------------------------------------
# 2.1) Dates exclues (LST manquante)
# ------------------------------------------------------------
EXCLUDED_DATES = set(date(2022, 10, d) for d in range(11, 23))

print("Dates exclues (LST manquante) :")
for d in sorted(EXCLUDED_DATES):
    print(" -", d)

## 3. Fonctions utilitaires

Adaptation Colab : les entrées de `name_files.txt` écrites sous Windows
(antislashs `\`, chemins absolus `C:\...\input_data_ML\...`) sont converties
automatiquement en chemins Drive par `_win_to_colab_path`.

In [ ]:
# ------------------------------------------------------------
# 3) Fonctions utilitaires
# ------------------------------------------------------------
def extract_date_from_filename(path):
    """
    Extrait YYYYMMDD depuis le nom de fichier.
    Exemple: Precip_20200101.tif, MODIS_LST_20200101.tif, etc.
    """
    name = os.path.basename(path)
    m = re.search(r"(\d{8})", name)
    if not m:
        raise ValueError(f"Impossible d'extraire la date depuis : {name}")
    return datetime.strptime(m.group(1), "%Y%m%d").date()


def _win_to_colab_path(p: str) -> str:
    """
    [COLAB] Convertit une entrée écrite sous Windows en chemin utilisable sur Colab (Linux):
    - antislashs -> slashs
    - chemin absolu Windows (C:/.../input_data_ML/xxx) -> xxx (relatif à BASE_DIR)
    Les chemins déjà valides (relatifs, ou absolus sous BASE_DIR) sont inchangés.
    """
    p = str(p).strip().replace("\\", "/")
    if re.match(r"^[A-Za-z]:/", p):
        root_name = os.path.basename(os.path.normpath(BASE_DIR)).lower()
        parts = p.split("/")
        low = [x.lower() for x in parts]
        if root_name in low:
            rest = parts[low.index(root_name) + 1:]
            p = "/".join(rest) if rest else "."
    return p


def parse_name_file(txt_path):
    """
    Parse name_files.txt en dict:
    { "SectionName": [relative/path1.tif, ...], ... }
    """
    sections = {}
    current = None
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("[") and line.endswith("]"):
                current = line[1:-1].strip()
                sections[current] = []
            else:
                if current is None:
                    continue
                sections[current].append(_win_to_colab_path(line))  # [COLAB] chemin Windows -> Colab
    return sections


def resolve_section(sections, preferred_names, keywords, prefer_contains=None):
    """
    Résout un nom de section de manière robuste.

    - preferred_names: str ou list[str] (ex: ["CHIRPS_Clipped","Precipitations_IDW"])
    - keywords: list[str] (match partiel dans nom section)
    - prefer_contains: str optionnel, pour prioriser un match (ex: "API" ou "IDW")

    Stratégie:
      1) match exact (case-insensitive) sur preferred_names
      2) match par keywords + score (IDW/API/len)
    """
    if isinstance(preferred_names, str):
        preferred_names = [preferred_names]

    # 1) exact match
    pref_lows = [p.lower() for p in preferred_names]
    for k in sections.keys():
        if k.lower() in pref_lows:
            return k

    # 2) keyword match + scoring
    candidates = []
    for k in sections.keys():
        kl = k.lower()
        if any(kw.lower() in kl for kw in keywords):
            score = 0
            # bonus si correspond à un "préféré" partiel
            for p in pref_lows:
                if p in kl:
                    score += 5
            # bonus IDW / API
            if "idw" in kl:
                score += 3
            if "api" in kl:
                score += 2
            # préférer un substring explicite si demandé
            if prefer_contains is not None and prefer_contains.lower() in kl:
                score += 10
            # petite pénalité pour noms très longs
            score -= 0.01 * len(k)
            candidates.append((score, k))

    if candidates:
        candidates.sort(reverse=True, key=lambda x: x[0])
        chosen = candidates[0][1]
        print(f"✅ Section résolue → '{chosen}' (keywords={keywords})")
        return chosen

    print("\nSections disponibles dans name_files.txt :")
    for k in sections.keys():
        print(" -", k)
    raise RuntimeError(
        f"Section introuvable (preferred={preferred_names}, keywords={keywords})."
    )


def build_date_to_path_dict_from_section(sections, section_name):
    """Construit un dict {date: abs_path} pour une section donnée."""
    if section_name not in sections:
        raise RuntimeError(f"Section '{section_name}' absente de {NAME_FILE}")

    d = {}
    for rel_path in sections[section_name]:
        abs_path = os.path.normpath(os.path.join(BASE_DIR, rel_path))
        if not os.path.isfile(abs_path):
            print(f"⚠️ Fichier introuvable (ignoré) : {abs_path}")
            continue
        try:
            date_key = extract_date_from_filename(abs_path)
            d[date_key] = abs_path
        except ValueError as e:
            print("⚠️", e)
    return d


def read_raster_raw(path):
    """Lit un raster bande 1 en float32, sans remplacer NoData."""
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
        nodata = src.nodata
    return arr, profile, nodata


def to_float_with_nan(arr, nodata):
    """Remplace nodata -> NaN."""
    arr = arr.astype("float32")
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    return arr


def reproject_to_template(
    src_arr,
    src_profile,
    template_profile,
    src_nodata=None,
    dst_nodata=np.nan,
    resampling=Resampling.average,
):
    """Rééchantillonne src_arr sur la grille template_profile en respectant NoData."""
    dst_arr = np.full(
        (template_profile["height"], template_profile["width"]),
        dst_nodata,
        dtype="float32",
    )
    kwargs = dict(
        source=src_arr,
        destination=dst_arr,
        src_transform=src_profile["transform"],
        src_crs=src_profile["crs"],
        dst_transform=template_profile["transform"],
        dst_crs=template_profile["crs"],
        dst_nodata=dst_nodata,
        resampling=resampling,
    )
    if src_nodata is not None:
        kwargs["src_nodata"] = src_nodata

    reproject(**kwargs)
    return dst_arr


def fill_nan_with_mean(arr):
    """Remplace NaN par la moyenne (si possible)."""
    if np.all(np.isnan(arr)):
        return arr
    m = np.nanmean(arr)
    return np.where(np.isnan(arr), m, arr)


def build_prev_date_support(date_dict):
    """Dernière date <= target_date (utile NDVI/ET)."""
    dates_sorted = sorted(date_dict.keys())

    def get_prev(target_date):
        if not dates_sorted:
            return None
        idx = bisect.bisect_right(dates_sorted, target_date) - 1
        if idx < 0:
            return None
        return dates_sorted[idx]

    return get_prev


# --- Masque Regions reconstruit à partir des rasters ---
REGION_MASK = None

def update_region_mask(precip_res, lst_res, ndvi_res, template_profile):
    global REGION_MASK
    base_mask = ~(np.isnan(precip_res) & np.isnan(lst_res) & np.isnan(ndvi_res))
    if REGION_MASK is None:
        REGION_MASK = base_mask
        print("\n🗺️ Masque 'Regions' initial construit à partir des rasters (1er jour).")
    else:
        REGION_MASK = REGION_MASK | base_mask

def get_region_mask(template_profile):
    global REGION_MASK
    if REGION_MASK is None:
        print("⚠️ REGION_MASK non initialisé, on prend toute la grille.")
        REGION_MASK = np.ones(
            (template_profile["height"], template_profile["width"]), dtype=bool
        )
    return REGION_MASK

### Contrôle des entrées de `name_files.txt` (ajout Colab)

Vérifie, avant le calcul long, que chaque entrée du fichier se résout bien en fichier présent sur Drive.
Une section en ❌ signale un chemin à corriger. Lecture seule ; peut prendre une à deux minutes sur Drive.

In [ ]:
# [COLAB] Contrôle rapide : les entrées de name_files.txt se résolvent-elles sur Drive ?
_sections_check = parse_name_file(NAME_FILE)
print(f"{len(_sections_check)} sections lues dans {os.path.basename(NAME_FILE)}\n")

for _sec, _entries in _sections_check.items():
    _missing = [_e for _e in _entries
                if not os.path.isfile(os.path.normpath(os.path.join(BASE_DIR, _e)))]
    _n_ok = len(_entries) - len(_missing)
    _flag = "✅" if (_n_ok > 0 and not _missing) else ("⚠️" if _n_ok > 0 else "❌")
    print(f"{_flag} [{_sec}] {len(_entries)} entrées -> {_n_ok} fichiers | introuvables: {len(_missing)}")
    for _e in _missing[:3]:
        print(f"      ex. introuvable: {os.path.normpath(os.path.join(BASE_DIR, _e))}")

## 4. Indexation des fichiers par date (via name_files.txt)

In [ ]:
# ------------------------------------------------------------
# 4) Indexation des fichiers par date (via name_files.txt)
# ------------------------------------------------------------
print("\n=== INDEXATION DES FICHIERS PAR DATE (via name_files.txt) ===")

if not os.path.isfile(NAME_FILE):
    raise RuntimeError(f"⚠️ Fichier introuvable : {NAME_FILE}")

sections = parse_name_file(NAME_FILE)

print("\nSections trouvées dans name_files.txt :")
for k in sections.keys():
    print(" -", k)

# ---- Résolution robuste des sections ----
# Precip (ex-CHIRPS)
if PREFER_API_PRECIP:
    SECTION_PRECIP = resolve_section(
        sections,
        preferred_names=["Precipitations_API_IDW", "Precipitations_IDW", "CHIRPS_Clipped"],
        keywords=["precip", "precipitations", "chirps", "rain", "pluie"],
        prefer_contains="api",
    )
else:
    SECTION_PRECIP = resolve_section(
        sections,
        preferred_names=["Precipitations_IDW", "Precipitations_API_IDW", "CHIRPS_Clipped"],
        keywords=["precip", "precipitations", "chirps", "rain", "pluie"],
        prefer_contains="idw",
    )

SECTION_LST    = resolve_section(sections, "MODIS_LST_Clipped",  ["lst"])
SECTION_NDVI   = resolve_section(sections, "MODIS_NDVI_Clipped", ["ndvi"])
SECTION_ET     = resolve_section(sections, "MODIS_ET_Clipped",   ["modis_et", " et"])
SECTION_GWS    = resolve_section(sections, "GLDAS_GWS_Clipped",  ["gws"])
SECTION_QSB    = resolve_section(sections, "ERA5_Qsb_Clipped",   ["qsb"])
SECTION_RZSM   = resolve_section(sections, "SMAP_RZSM_Clipped",  ["rzsm"])
SECTION_SOILM  = resolve_section(sections, "SMAP_SoilM_Clipped", ["soilm", "soil"])

print("\n✅ Sections utilisées :")
print(" - PRECIP :", SECTION_PRECIP)
print(" - LST    :", SECTION_LST)
print(" - NDVI   :", SECTION_NDVI)
print(" - ET     :", SECTION_ET)
print(" - GWS    :", SECTION_GWS)
print(" - QSB    :", SECTION_QSB)
print(" - RZSM   :", SECTION_RZSM)
print(" - SOILM  :", SECTION_SOILM)

precip_dict = build_date_to_path_dict_from_section(sections, SECTION_PRECIP)
lst_dict    = build_date_to_path_dict_from_section(sections, SECTION_LST)
ndvi_dict   = build_date_to_path_dict_from_section(sections, SECTION_NDVI)
et_dict     = build_date_to_path_dict_from_section(sections, SECTION_ET)
gws_dict    = build_date_to_path_dict_from_section(sections, SECTION_GWS)
qsb_dict    = build_date_to_path_dict_from_section(sections, SECTION_QSB)
rzsm_dict   = build_date_to_path_dict_from_section(sections, SECTION_RZSM)
soilm_dict  = build_date_to_path_dict_from_section(sections, SECTION_SOILM)

print("\n--- Résumé des dates disponibles ---")
print("Nb dates PRECIP :", len(precip_dict))
print("Nb dates LST    :", len(lst_dict))
print("Nb dates NDVI   :", len(ndvi_dict))
print("Nb dates ET     :", len(et_dict))
print("Nb dates GWS    :", len(gws_dict))
print("Nb dates Qsb    :", len(qsb_dict))
print("Nb dates RZSM   :", len(rzsm_dict))
print("Nb dates SoilM  :", len(soilm_dict))

def _print_minmax(dct, label):
    if dct:
        print(f"{label:<7} : {min(dct)} → {max(dct)}")

_print_minmax(precip_dict, "PRECIP")
_print_minmax(lst_dict,    "LST")
_print_minmax(ndvi_dict,   "NDVI")
_print_minmax(et_dict,     "ET")
_print_minmax(gws_dict,    "GWS")
_print_minmax(qsb_dict,    "QSB")
_print_minmax(rzsm_dict,   "RZSM")
_print_minmax(soilm_dict,  "SOILM")

get_ndvi_date_for = build_prev_date_support(ndvi_dict)
get_et_date_for   = build_prev_date_support(et_dict)

## 5. Construction des dates TRAIN / TEST

In [ ]:
# ------------------------------------------------------------
# 5) Construction des dates TRAIN / TEST
# ------------------------------------------------------------
print("\n=== CONSTRUCTION DES DATES TRAIN / TEST (5% des dates pour TEST) ===")

all_common_dates = sorted(
    set(precip_dict.keys())
    & set(lst_dict.keys())
    & set(gws_dict.keys())
    & set(qsb_dict.keys())
    & set(rzsm_dict.keys())
    & set(soilm_dict.keys())
)

print("Nombre total de dates communes (PRECIP ∩ LST ∩ GWS ∩ Qsb ∩ RZSM ∩ SoilM) :", len(all_common_dates))
if not all_common_dates:
    raise RuntimeError("⚠️ Aucune date commune entre les variables journalières.")

date_candidates = [
    d for d in all_common_dates
    if (TRAIN_START <= d <= TRAIN_END) and (d not in EXCLUDED_DATES)
]
if not date_candidates:
    raise RuntimeError("⚠️ Aucune date candidate dans la fenêtre TRAIN (après exclusion).")

train_dates, test_dates = train_test_split(
    date_candidates,
    test_size=TEST_FRACTION,
    random_state=42,
    shuffle=True,
)
train_dates = sorted(train_dates)
test_dates  = sorted(test_dates)

print(f"\nFenêtre TRAIN (pool) : {TRAIN_START} → {TRAIN_END}")
print(f" → Nombre de dates train : {len(train_dates)}")
print(" → Première date train :", min(train_dates))
print(" → Dernière date train :", max(train_dates))

print(f"\nTEST = 5% des dates dans la même fenêtre ({TRAIN_START} → {TRAIN_END})")
print(f" → Nombre de dates test : {len(test_dates)}")
print(" → Première date test :", min(test_dates))
print(" → Dernière date test :", max(test_dates))

## 6. Construction dataset (pixels valides)

In [ ]:
# ------------------------------------------------------------
# 6) Construction dataset (pixels valides)
# ------------------------------------------------------------
print("\n=== CONSTRUCTION DU DATASET (toutes les dates TRAIN + TEST) ===")

X_train_list, y_train_list = [], []
X_test_list,  y_test_list  = [], []

template_profile = None

def build_Xy_for_date(date_key, template_profile, role="train"):
    ndvi_date = get_ndvi_date_for(date_key)
    if ndvi_date is None:
        print(f"⏭️ [{role}] {date_key} : aucun NDVI (<= date), ignoré.")
        return None, None, template_profile

    et_date = get_et_date_for(date_key)
    if et_date is None:
        print(f"⏭️ [{role}] {date_key} : aucun ET (<= date), ignoré.")
        return None, None, template_profile

    if date_key not in gws_dict:
        print(f"⏭️ [{role}] {date_key} : GWS manquant, ignoré.")
        return None, None, template_profile

    gws_raw, gws_prof, gws_nodata = read_raster_raw(gws_dict[date_key])
    gws_arr = to_float_with_nan(gws_raw, gws_nodata)

    if template_profile is None:
        template_profile = gws_prof
        print("\nGrille GLDAS (template) :")
        print(" - CRS :", template_profile["crs"])
        print(" - Taille :", template_profile["width"], "x", template_profile["height"])
        print(" - Résolution approx (deg) :", template_profile["transform"][0])

    required = (
        (date_key in precip_dict)
        and (date_key in lst_dict)
        and (date_key in qsb_dict)
        and (date_key in rzsm_dict)
        and (date_key in soilm_dict)
    )
    if not required:
        print(f"⏭️ [{role}] {date_key} : variable journalière manquante, ignoré.")
        return None, None, template_profile

    precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile,
                                       src_nodata=precip_nodata, dst_nodata=np.nan)
    lst_res    = reproject_to_template(lst_raw, lst_prof, template_profile,
                                       src_nodata=lst_nodata, dst_nodata=np.nan)
    ndvi_res   = reproject_to_template(ndvi_raw, ndvi_prof, template_profile,
                                       src_nodata=ndvi_nodata, dst_nodata=np.nan)
    et_res     = reproject_to_template(et_raw, et_prof, template_profile,
                                       src_nodata=et_nodata, dst_nodata=np.nan)
    qsb_res    = reproject_to_template(qsb_raw, qsb_prof, template_profile,
                                       src_nodata=qsb_nodata, dst_nodata=np.nan)
    rzsm_res   = reproject_to_template(rzsm_raw, rzsm_prof, template_profile,
                                       src_nodata=rzsm_nodata, dst_nodata=np.nan)
    soilm_res  = reproject_to_template(soilm_raw, soilm_prof, template_profile,
                                       src_nodata=soilm_nodata, dst_nodata=np.nan)

    update_region_mask(precip_res, lst_res, ndvi_res, template_profile)
    region_mask = get_region_mask(template_profile)

    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    mask_valid = (~np.isnan(gws_arr)) & region_mask
    if not np.any(mask_valid):
        print(f"⏭️ [{role}] {date_key} : aucun pixel valide (GWS/Regions), ignoré.")
        return None, None, template_profile

    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2 * np.pi * doy / 365.0)
    cos_doy = np.cos(2 * np.pi * doy / 365.0)

    n_pixels = np.count_nonzero(mask_valid)
    sin_plane = np.full(n_pixels, sin_doy, dtype="float32")
    cos_plane = np.full(n_pixels, cos_doy, dtype="float32")

    X_day = np.stack(
        [
            precip_f[mask_valid],
            lst_f[mask_valid],
            ndvi_f[mask_valid],
            et_f[mask_valid],
            qsb_f[mask_valid],
            rzsm_f[mask_valid],
            soilm_f[mask_valid],
            sin_plane,
            cos_plane,
        ],
        axis=1,
    )
    y_day = gws_arr[mask_valid]

    row_mask = (~np.isnan(y_day)) & (~np.isnan(X_day).any(axis=1))
    if not np.any(row_mask):
        print(f"⏭️ [{role}] {date_key} : NaN dans features/cible, ignoré.")
        return None, None, template_profile

    X_day = X_day[row_mask]
    y_day = y_day[row_mask]

    print(f" ✔️ [{role}] {date_key} : {X_day.shape[0]} pixels utilisés.")
    return X_day, y_day, template_profile


for d in train_dates:
    X_day, y_day, template_profile = build_Xy_for_date(d, template_profile, role="train")
    if X_day is None:
        continue
    X_train_list.append(X_day)
    y_train_list.append(y_day)

for d in test_dates:
    X_day, y_day, template_profile = build_Xy_for_date(d, template_profile, role="test")
    if X_day is None:
        continue
    X_test_list.append(X_day)
    y_test_list.append(y_day)

if not X_train_list:
    raise RuntimeError("⚠️ Aucun pixel d'entraînement valide n'a été trouvé.")
if not X_test_list:
    raise RuntimeError("⚠️ Aucun pixel de test valide n'a été trouvé.")

X_train = np.vstack(X_train_list).astype("float32")
y_train = np.concatenate(y_train_list).astype("float32")
X_test  = np.vstack(X_test_list).astype("float32")
y_test  = np.concatenate(y_test_list).astype("float32")

print(f"\nNombre total de pixels d'entraînement : {X_train.shape[0]}")
print(f"Nombre total de pixels de test        : {X_test.shape[0]}")
print("Exemple feature train [P,LST,NDVI,ET,Qsb,RZSM,SoilM,sinDOY,cosDOY] :")
print(" ", X_train[0])

## 7. Modèle HuberRegressor + StandardScaler

In [ ]:
# ------------------------------------------------------------
# 7) Modèle HuberRegressor + StandardScaler
# ------------------------------------------------------------
print("\n=== ENTRAÎNEMENT DU MODÈLE HUBERREGRESSOR ===")

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, shuffle=True
)
print(f"Taille X_tr : {X_tr.shape}, X_val : {X_val.shape}")

huber = HuberRegressor(
    epsilon=1.35,
    alpha=0.0001,
    max_iter=1000,
    tol=1e-4,
    fit_intercept=True,
    warm_start=False,
)

huber_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("huber", huber),
])

print("\n🔄 Entraînement HuberRegressor...")
t0 = time.time()
huber_model.fit(X_tr, y_tr)
t1 = time.time()
training_time_ms = (t1 - t0) * 1000.0

y_tr_pred  = huber_model.predict(X_tr)
y_val_pred = huber_model.predict(X_val)
y_test_pred = huber_model.predict(X_test)

def _metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return r2, mae, rmse

r2_tr, mae_tr, rmse_tr = _metrics(y_tr, y_tr_pred)
r2_val, mae_val, rmse_val = _metrics(y_val, y_val_pred)
r2_test, mae_test, rmse_test = _metrics(y_test, y_test_pred)

print("\n╔═════════ MODÈLE HUBERREGRESSOR ═════════╗")
print(f" ➤ Temps d'entraînement : {training_time_ms:.2f} ms")
print(" --- TRAIN (X_tr) ---")
print(f"    R² TRAIN  : {r2_tr:.4f}")
print(f"    MAE TRAIN : {mae_tr:.4f}")
print(f"    RMSE TRAIN: {rmse_tr:.4f}")
print(" --- VALID (X_val) ---")
print(f"    R² VAL    : {r2_val:.4f}")
print(f"    MAE VAL   : {mae_val:.4f}")
print(f"    RMSE VAL  : {rmse_val:.4f}")
print(" --- TEST (pixels sur dates 5%) ---")
print(f"    R² TEST   : {r2_test:.4f}")
print(f"    MAE TEST  : {mae_test:.4f}")
print(f"    RMSE TEST : {rmse_test:.4f}")
print("╚══════════════════════════════════╝")

## 8. Évaluation cartographique (dates TEST) + sauvegarde

In [ ]:
# ------------------------------------------------------------
# 8) Évaluation cartographique (dates TEST) + sauvegarde
# ------------------------------------------------------------
print("\n=== ÉVALUATION CARTOGRAPHIQUE SUR LES DATES TEST ===")

region_mask = get_region_mask(template_profile)
all_true, all_pred = [], []

for date_key in test_dates:
    print(f"\n🧪 Test cartographique : {date_key}")

    ndvi_date = get_ndvi_date_for(date_key)
    et_date   = get_et_date_for(date_key)
    if ndvi_date is None or et_date is None:
        print(" ⏭️ NDVI/ET indisponible, ignoré.")
        continue
    if date_key not in gws_dict:
        print(" ⏭️ GWS manquant, ignoré.")
        continue

    gws_raw, _, gws_nodata = read_raster_raw(gws_dict[date_key])
    gws_arr = to_float_with_nan(gws_raw, gws_nodata)

    if not ((date_key in precip_dict) and (date_key in lst_dict) and (date_key in qsb_dict) and (date_key in rzsm_dict) and (date_key in soilm_dict)):
        print(" ⏭️ variable journalière manquante, ignoré.")
        continue

    precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata, dst_nodata=np.nan)
    lst_res    = reproject_to_template(lst_raw, lst_prof, template_profile, src_nodata=lst_nodata, dst_nodata=np.nan)
    ndvi_res   = reproject_to_template(ndvi_raw, ndvi_prof, template_profile, src_nodata=ndvi_nodata, dst_nodata=np.nan)
    et_res     = reproject_to_template(et_raw, et_prof, template_profile, src_nodata=et_nodata, dst_nodata=np.nan)
    qsb_res    = reproject_to_template(qsb_raw, qsb_prof, template_profile, src_nodata=qsb_nodata, dst_nodata=np.nan)
    rzsm_res   = reproject_to_template(rzsm_raw, rzsm_prof, template_profile, src_nodata=rzsm_nodata, dst_nodata=np.nan)
    soilm_res  = reproject_to_template(soilm_raw, soilm_prof, template_profile, src_nodata=soilm_nodata, dst_nodata=np.nan)

    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2 * np.pi * doy / 365.0)
    cos_doy = np.cos(2 * np.pi * doy / 365.0)

    n_pixels_full = precip_f.size
    sin_plane_full = np.full(n_pixels_full, sin_doy, dtype="float32")
    cos_plane_full = np.full(n_pixels_full, cos_doy, dtype="float32")

    X_all = np.stack(
        [
            precip_f.ravel(),
            lst_f.ravel(),
            ndvi_f.ravel(),
            et_f.ravel(),
            qsb_f.ravel(),
            rzsm_f.ravel(),
            soilm_f.ravel(),
            sin_plane_full,
            cos_plane_full,
        ],
        axis=1,
    )

    if np.isnan(X_all).any():
        print(" ⏭️ NaN dans X_all, ignoré.")
        continue

    y_pred_all = huber_model.predict(X_all).astype("float32")
    pred_full = y_pred_all.reshape(template_profile["height"], template_profile["width"])

    pred_clip = pred_full.copy()
    pred_clip[~region_mask] = NODATA_VALUE

    mask_gws = (~np.isnan(gws_arr)) & region_mask
    if not np.any(mask_gws):
        print(" ⚠️ aucun pixel valide GWS, ignoré.")
        continue

    y_true = gws_arr[mask_gws]
    y_pred = pred_clip[mask_gws]

    r2_d, mae_d, rmse_d = _metrics(y_true, y_pred)
    print(f" ➤ R²   : {r2_d:.4f}")
    print(f" ➤ MAE  : {mae_d:.4f}")
    print(f" ➤ RMSE : {rmse_d:.4f}")

    all_true.append(y_true)
    all_pred.append(y_pred)

    out_profile_test = template_profile.copy()
    out_profile_test.update(dtype="float32", count=1, nodata=NODATA_VALUE)

    out_name_test = os.path.join(OUT_DIR, f"GWS_TEST_PRED_HUBER_{date_key.strftime('%Y%m%d')}.tif")
    with rasterio.open(out_name_test, "w", **out_profile_test) as dst:
        dst.write(pred_clip, 1)
    print(f" ✔️ Carte test sauvegardée : {out_name_test}")

if all_true:
    all_true_flat = np.concatenate(all_true)
    all_pred_flat = np.concatenate(all_pred)
    r2_g, mae_g, rmse_g = _metrics(all_true_flat, all_pred_flat)
    print("\n📊 MÉTRIQUES GLOBALES (toutes dates TEST)")
    print(f" ➤ R² global   : {r2_g:.4f}")
    print(f" ➤ MAE global  : {mae_g:.4f}")
    print(f" ➤ RMSE global : {rmse_g:.4f}")
else:
    print("\n⚠️ Aucune métrique globale (aucun pixel valide).")

## 9. Forecast cartographique (sans trous, clip)

In [ ]:
# ------------------------------------------------------------
# 9) Forecast cartographique (sans trous, clip)
# ------------------------------------------------------------
print("\n=== PHASE DE PRÉDICTION CARTOGRAPHIQUE (FORECAST) ===")

if template_profile is None:
    raise RuntimeError("Template GLDAS introuvable (aucun GWS lu).")

out_profile = template_profile.copy()
out_profile.update(dtype="float32", count=1, nodata=NODATA_VALUE)

all_forecast_dates = sorted(
    set(precip_dict.keys())
    & set(lst_dict.keys())
    & set(qsb_dict.keys())
    & set(rzsm_dict.keys())
    & set(soilm_dict.keys())
)

if not all_forecast_dates:
    print("⚠️ Aucune date commune PRECIP ∩ LST ∩ Qsb ∩ RZSM ∩ SoilM. Impossible de prédire.")
else:
    print("Plage possible forecast :", min(all_forecast_dates), "→", max(all_forecast_dates))

user_start = FORECAST_START.date()
user_end   = FORECAST_END.date()

forecast_start_date = max(user_start, min(all_forecast_dates)) if all_forecast_dates else user_start
forecast_end_date   = min(user_end,   max(all_forecast_dates)) if all_forecast_dates else user_end

print("Fenêtre demandée :", user_start, "→", user_end)
print("Fenêtre utilisée :", forecast_start_date, "→", forecast_end_date)

forecast_dates = [
    d for d in all_forecast_dates
    if (forecast_start_date <= d <= forecast_end_date) and (d not in EXCLUDED_DATES)
]

print(f"Nombre de dates éligibles forecast : {len(forecast_dates)}")
if forecast_dates:
    print("Première :", min(forecast_dates))
    print("Dernière :", max(forecast_dates))

region_mask = get_region_mask(template_profile)

for date_key in forecast_dates:
    ndvi_date = get_ndvi_date_for(date_key)
    et_date   = get_et_date_for(date_key)
    if ndvi_date is None or et_date is None:
        print(f"⏭️ {date_key} : NDVI/ET indisponible, ignorée.")
        continue

    precip_raw, precip_prof, precip_nodata = read_raster_raw(precip_dict[date_key])
    lst_raw,    lst_prof,    lst_nodata    = read_raster_raw(lst_dict[date_key])
    ndvi_raw,   ndvi_prof,   ndvi_nodata   = read_raster_raw(ndvi_dict[ndvi_date])
    et_raw,     et_prof,     et_nodata     = read_raster_raw(et_dict[et_date])
    qsb_raw,    qsb_prof,    qsb_nodata    = read_raster_raw(qsb_dict[date_key])
    rzsm_raw,   rzsm_prof,   rzsm_nodata   = read_raster_raw(rzsm_dict[date_key])
    soilm_raw,  soilm_prof,  soilm_nodata  = read_raster_raw(soilm_dict[date_key])

    precip_res = reproject_to_template(precip_raw, precip_prof, template_profile, src_nodata=precip_nodata, dst_nodata=np.nan)
    lst_res    = reproject_to_template(lst_raw, lst_prof, template_profile, src_nodata=lst_nodata, dst_nodata=np.nan)
    ndvi_res   = reproject_to_template(ndvi_raw, ndvi_prof, template_profile, src_nodata=ndvi_nodata, dst_nodata=np.nan)
    et_res     = reproject_to_template(et_raw, et_prof, template_profile, src_nodata=et_nodata, dst_nodata=np.nan)
    qsb_res    = reproject_to_template(qsb_raw, qsb_prof, template_profile, src_nodata=qsb_nodata, dst_nodata=np.nan)
    rzsm_res   = reproject_to_template(rzsm_raw, rzsm_prof, template_profile, src_nodata=rzsm_nodata, dst_nodata=np.nan)
    soilm_res  = reproject_to_template(soilm_raw, soilm_prof, template_profile, src_nodata=soilm_nodata, dst_nodata=np.nan)

    precip_f = fill_nan_with_mean(precip_res)
    lst_f    = fill_nan_with_mean(lst_res)
    ndvi_f   = fill_nan_with_mean(ndvi_res)
    et_f     = fill_nan_with_mean(et_res)
    qsb_f    = fill_nan_with_mean(qsb_res)
    rzsm_f   = fill_nan_with_mean(rzsm_res)
    soilm_f  = fill_nan_with_mean(soilm_res)

    doy = date_key.timetuple().tm_yday
    sin_doy = np.sin(2 * np.pi * doy / 365.0)
    cos_doy = np.cos(2 * np.pi * doy / 365.0)

    n_pixels_full = precip_f.size
    sin_plane_full = np.full(n_pixels_full, sin_doy, dtype="float32")
    cos_plane_full = np.full(n_pixels_full, cos_doy, dtype="float32")

    X_all = np.stack(
        [
            precip_f.ravel(),
            lst_f.ravel(),
            ndvi_f.ravel(),
            et_f.ravel(),
            qsb_f.ravel(),
            rzsm_f.ravel(),
            soilm_f.ravel(),
            sin_plane_full,
            cos_plane_full,
        ],
        axis=1,
    )

    if np.isnan(X_all).any():
        print(f"⏭️ {date_key} : NaN dans X_all, ignorée.")
        continue

    y_pred_all = huber_model.predict(X_all).astype("float32")
    pred_full = y_pred_all.reshape(template_profile["height"], template_profile["width"])

    pred_clip = pred_full.copy()
    pred_clip[~region_mask] = NODATA_VALUE

    out_name = os.path.join(OUT_DIR, f"GWS_PRED_HUBER_{date_key.strftime('%Y%m%d')}.tif")
    with rasterio.open(out_name, "w", **out_profile) as dst:
        dst.write(pred_clip, 1)

    print(f" ✔️ Forecast sauvegardé : {out_name}")

# Récapitulatif
if os.path.isdir(OUT_DIR):
    tif_files = [f for f in os.listdir(OUT_DIR) if f.lower().endswith(".tif")]
    print(f"\n📂 Nombre total de cartes .tif dans {OUT_DIR} : {len(tif_files)}")
    if not tif_files:
        print("⚠️ Aucune image .tif trouvée.")
else:
    print(f"⚠️ Le dossier de sortie {OUT_DIR} n'existe pas.")

## Récapitulatif des sorties (ajout Colab)

In [ ]:
# [COLAB] Nombre de fichiers présents dans le dossier de sortie, par extension
from collections import Counter

_count = Counter(os.path.splitext(_f)[1].lower() or "(sans extension)" for _f in os.listdir(OUT_DIR))
for _ext, _n in sorted(_count.items()):
    print(f"{_n:6d} fichier(s) {_ext}")
print("Dossier de sortie :", OUT_DIR)